# S1.2 · Counting and tables

*Phase S, statistics and probability · Part 1, describing data · lesson 2 of 10 · about 2 hours*

Counting is the oldest statistic, and still the most used. Most of what a bibliometric study reports is a count: articles per year, references per type, authors per country. This lesson turns counts into tables and pictures. It covers frequency tables, the mode, bar charts that tell the truth, two-way tables, and percentages that can be read in two directions, which is where many published claims go wrong.

**Run this lesson:** [in your browser (JupyterLite)](https://sbrzt.github.io/personal-course/lite/lab/index.html?path=lessons/phase-s/s1-2-counting-and-tables.ipynb) · [in Google Colab](https://colab.research.google.com/github/sbrzt/personal-course/blob/main/lessons/phase-s/s1-2-counting-and-tables.ipynb) · on your computer (see *How this course works* on the course site)

**By the end you can**

- build a frequency table with counts, shares and cumulative shares;
- find the mode, and say when it is a useful summary;
- draw a bar chart that does not mislead;
- cross two variables in a two-way table, with its totals;
- choose between row and column percentages, and read each one correctly;
- count references by type, knowing what the count depends on.

Exercises, hints, solutions and sliders work as in the earlier lessons. Run the setup cell first.

In [ ]:
# ⚙️ Setup: run this cell first
import sys, pathlib, subprocess

if sys.platform == "emscripten":            # in the browser (JupyterLite)
    import piplite
    await piplite.install(["personal-course"])
elif "google.colab" in sys.modules:         # in Google Colab
    repo = "git+https://github.com/sbrzt/personal-course"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    f"personal-course @ {repo}"], check=True)
else:                                       # on your computer
    here = pathlib.Path.cwd().resolve()
    for folder in [here, *here.parents]:    # find the course folder
        if (folder / "course" / "__init__.py").exists():
            sys.path.insert(0, str(folder))
            break

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from course import bibliothecae, data, plots, stats
from course.widgets import explore
from course.checks import s1_2 as check

plots.use_style()
bib = data.bibliothecae()          # the seven tables of the journal's data
print("Ready.")

## 1 · Frequency tables

Ten books come back to a library on a Monday: 5 novels, 3 essays and 2 books of poetry. Sorting them into piles and counting each pile gives a **frequency table**. Each count is an **absolute frequency**. Divided by the total, each count becomes a **relative frequency**, or share:

$$f_k = \frac{n_k}{n}$$

| Symbol | Read it as | Means |
| --- | --- | --- |
| $k$ | "k" | one of the categories: here, novel, essay or poetry |
| $K$ | "capital K" | the number of categories: here 3 |
| $n_k$ | "n k" | the count in category k: its absolute frequency |
| $n$ | "n" | the total count: here 10 |
| $f_k$ | "f k" | the share of category k: its relative frequency |

Read aloud: *"f k equals n k over n."* **By hand:** the novels' share is $5/10 = 0.5$, the essays' $3/10 = 0.3$ and the poetry's $2/10 = 0.2$. Multiplied by 100, the shares are percentages: 50%, 30% and 20%. The shares always add up to 1, because the counts add up to the total:

$$\sum_{k=1}^{K} f_k = \frac{n_1 + n_2 + \dots + n_K}{n} = 1$$

**In code**, `value_counts()` counts each category and lists them from the largest to the smallest; `value_counts(normalize=True)` gives the shares instead.

In [ ]:
returns = pd.Series(["novel"] * 5 + ["essay"] * 3 + ["poetry"] * 2)
pd.DataFrame({"count": returns.value_counts(),
              "share": returns.value_counts(normalize=True)})

**In the data.** The 321 articles of *Bibliothecae\.it* by language:

In [ ]:
languages = bib.articles["langLabel"]
pd.DataFrame({"articles": languages.value_counts(),
              "share": languages.value_counts(normalize=True).round(3)})

Nine articles in ten are in Italian. With one category so large, the others are easier to compare as counts than as shares: 18 in English, 9 in Portuguese, 2 each in French and German.

### ✏️ Exercise 1 · Articles by section

The column `bib.articles["section"]` gives each article's section. Compute `counts`, the number of articles in each section; `shares`, the same as shares of all articles; and `share_top3`, the share of all articles that belong to the three largest sections taken together.

In [ ]:
sections = bib.articles["section"]
counts = ...
shares = ...
share_top3 = ...

check.ex1(counts, shares, share_top3)

<details>
<summary>💡 Hint</summary>

`value_counts()` and `value_counts(normalize=True)` give the two columns, already sorted from the largest section to the smallest. The first three shares are `shares.iloc[:3]`: add them up with `.sum()`.

</details>

<details>
<summary>✅ Solution</summary>

```python
sections = bib.articles["section"]
counts = sections.value_counts()
shares = sections.value_counts(normalize=True)
share_top3 = shares.iloc[:3].sum()

check.ex1(counts, shares, share_top3)
```

</details>

## 2 · Cumulative frequencies

When the categories have an order, such as years, a running total answers questions of the form "how many up to here?". Adding up the shares from the first category to category k gives the **cumulative share**:

$$F_k = \sum_{j=1}^{k} f_j = f_1 + f_2 + \dots + f_k$$

| Symbol | Read it as | Means |
| --- | --- | --- |
| $j$ | "j" | a counter that runs over the categories up to k |
| $F_k$ | "capital F k" | the share of all observations in categories 1 to k |

Read aloud: *"capital F k is the sum of the shares f j, for j from 1 to k."* The last cumulative share is always 1. **By hand:** three years with shares 0.2, 0.3 and 0.5 have cumulative shares 0.2, 0.5 and 1.0, so half of everything had happened by the end of the second year.

A cumulative share only makes sense when the order means something. "The share of articles published up to 2018" is a real question; "the share of articles up to the section *Essays*" is not, because sections have no order (S1.1).

**In the data.** Articles per year, with their cumulative share:

In [ ]:
year = bib.articles["date"].str[:4].astype(int).rename("year")
by_year = year.value_counts().sort_index()          # sort by year, not by count
fig, axes = plt.subplots(1, 2, figsize=(9, 3.2))
axes[0].bar(by_year.index, by_year.to_numpy(), color=plots.BLUE)
axes[0].set(xlabel="year", ylabel="articles", title="articles each year")
axes[1].plot(by_year.index, by_year.cumsum() / by_year.sum(), "o-",
             color=plots.BLUE)
axes[1].axhline(0.5, color=plots.MUTED, ls="--")
axes[1].set(xlabel="year", ylabel="cumulative share", ylim=(0, 1.05),
            title="share published so far")
fig.tight_layout()
plt.show()

`value_counts()` sorts by count; for years, `.sort_index()` puts them back in time order, which a cumulative share needs.

### ✏️ Exercise 2 · Half of the journal

Compute `per_year`, the number of articles published each year in time order; `cum_share`, the share of all articles published up to and including each year; and `half_by`, the first year by the end of which at least half of the articles had appeared.

In [ ]:
per_year = ...
cum_share = ...
half_by = ...

check.ex2(per_year, cum_share, half_by)

<details>
<summary>💡 Hint</summary>

`year.value_counts().sort_index()` gives the counts in time order, and `.cumsum()` adds them up as it goes; divide by the total. `cum_share[cum_share >= 0.5]` keeps the years at or above one half, and `.index[0]` picks the first of them.

</details>

<details>
<summary>✅ Solution</summary>

```python
per_year = year.value_counts().sort_index()
cum_share = per_year.cumsum() / per_year.sum()
half_by = cum_share[cum_share >= 0.5].index[0]

check.ex2(per_year, cum_share, half_by)
```

</details>

## 3 · The mode

The most frequent category is called the **mode**. Of the returned books, the mode is "novel"; of the journal's languages, "Italian". For a nominal variable the mode is the only possible summary of the "typical" value: no average of languages exists.

Mathematics writes the mode with a new symbol, *argmax*:

$$\text{mode} = \underset{k}{\arg\max}\; n_k$$

Read aloud: *"the mode is the category k at which n k is largest."* The **max** of the counts is the largest count, 5; the **argmax** is *where* it is found, the category "novel". The difference between a largest value and where it is found will matter again: a classifier's answer is the argmax of its scores (see the AI note below).

The mode has two weaknesses:

- **It can be fragile.** Two categories with nearly the same count can swap places when a single observation changes.
- **It says little when there are many values.** A numeric variable can take so many values that its most common one is hardly more common than the rest.

**In code**, `value_counts().idxmax()` gives the argmax and `value_counts().max()` the count it has; `.mode()` lists all the categories that tie for first place.

In [ ]:
first_last = bib.articles["pages"].str.split("-", expand=True).astype(int)
pages = first_last[1] - first_last[0] + 1             # as in S1.1
print("mode of the language:", languages.mode().tolist())
print(f"{pages.nunique()} different page counts among {len(pages)} articles")

### ✏️ Exercise 3 · Modes

Compute `mode_year`, the year in which the journal published the most articles, and `n_mode_year`, how many it published that year. Then compute `mode_pages`, the most common number of pages, and `n_mode_pages`, how many articles have exactly that number of pages.

In [ ]:
mode_year = ...
n_mode_year = ...
mode_pages = ...
n_mode_pages = ...

check.ex3(mode_year, n_mode_year, mode_pages, n_mode_pages)

<details>
<summary>💡 Hint</summary>

`year.value_counts()` lists the years from the busiest to the quietest: `.idxmax()` gives the busiest year and `.max()` its count. Do the same with `pages`.

</details>

<details>
<summary>✅ Solution</summary>

```python
mode_year = year.value_counts().idxmax()
n_mode_year = year.value_counts().max()
mode_pages = pages.value_counts().idxmax()
n_mode_pages = pages.value_counts().max()

check.ex3(mode_year, n_mode_year, mode_pages, n_mode_pages)
```

The most common length, 6 pages, belongs to only 15 of the 321 articles, because there are 76 different lengths. For a number with so many values the mode is a poor summary; S1.4 introduces better ones.

</details>

## 4 · Bar charts that tell the truth

A bar chart draws each count as a bar whose **length** is proportional to the count. That is its whole point, and two rules follow from it:

- **The bars start at zero.** If they do not, their lengths no longer match the counts.
- **The order of the bars should help the reader.** Nominal categories are easiest to read sorted by size; ordered categories, such as years, keep their own order. Long labels fit better on horizontal bars.

The journal published 29 articles in 2023 and 24 in 2024. Move the slider to start the vertical axis higher, and watch the same two counts tell a different story:

In [ ]:
two_years = by_year.loc[[2023, 2024]]

def axis_start(start):
    fig, ax = plt.subplots(figsize=(5, 3.2))
    ax.bar(two_years.index.astype(str), two_years.to_numpy(), color=plots.BLUE)
    ax.set(ylim=(start, 30), ylabel="articles",
           title=f"vertical axis starts at {start}")
    plt.show()
    a, b = two_years.to_numpy() - start
    print(f"The 2024 bar looks {b / a:.0%} as tall as the 2023 bar; "
          f"the true ratio is {two_years[2024] / two_years[2023]:.0%}.")

explore(axis_start, start=(0, 23, 1, 0, "axis starts at"))

With the axis starting at 23, the 2024 bar looks a sixth as tall as the 2023 bar, although 2024 had 83% as many articles. A cut axis is acceptable for a line that shows changes over time, where the reader compares positions rather than lengths, but never for bars.

**Why bars rather than pies?** In a classic experiment, William Cleveland and Robert McGill asked people to compare values drawn as bars and as pie slices. People were more accurate with the bars in almost every case: "In only 3 of the 40 cases was the pie chart more accurate on average than the bar chart" ([Cleveland & McGill, 1984](https://www.jstor.org/stable/2288400)). Judging the position of the end of a bar is easier than judging an angle. Here are the sections, sorted, on horizontal bars:

In [ ]:
fig, ax = plt.subplots(figsize=(6.4, 3.2))
sections.value_counts().sort_values().plot.barh(ax=ax, color=plots.BLUE)
ax.set(xlabel="articles", ylabel="")
plt.show()

`sort_values()` puts the smallest first, and `barh` draws the first bar at the bottom, so the largest section ends up on top.

## 5 · Two-way tables

A frequency table counts one variable. A **two-way table**, also called a **contingency table** or **cross-tabulation**, counts two variables together: each cell holds the number of observations that fall in one category of the first variable *and* one category of the second.

| | essay | other section | total |
| --- | --- | --- | --- |
| **Italian** | 142 | 148 | 290 |
| **other language** | 27 | 4 | 31 |
| **total** | 169 | 152 | 321 |

The totals in the last row and column are called the **margins**, because they are written in the margins of the table. Statistics names the cells and the margins with two subscripts and a dot:

| Symbol | Read it as | Means | Here |
| --- | --- | --- | --- |
| $n_{ij}$ | "n i j" | the count in row i and column j | $n_{21} = 27$ essays not in Italian |
| $n_{i\cdot}$ | "n i dot" | the total of row i: the dot marks the index added over | $n_{2\cdot} = 31$ articles not in Italian |
| $n_{\cdot j}$ | "n dot j" | the total of column j | $n_{\cdot 1} = 169$ essays |
| $n$ | "n" | the total of the whole table | 321 |

So $n_{i\cdot} = \sum_j n_{ij}$: the total of a row adds up the cells of that row. **By hand:** $n_{1\cdot} = 142 + 148 = 290$, and $n_{\cdot 2} = 148 + 4 = 152$.

**In code**, `pd.crosstab(rows, columns)` builds the table, and `margins=True` adds the totals, labelled "All":

In [ ]:
period = pd.Series(np.where(year <= 2018, "2012-2018", "2019-2025"),
                   index=year.index, name="period")
pd.crosstab(sections, period, margins=True)

The two periods have the same number of years, but the second has more articles. Essays doubled, from 55 to 114; *Voices of living libraries* appears only in the second period, and *Causeries*, *Chapters* and *Profiles* only in the first.

### ✏️ Exercise 4 · A two-way table

Build `table`, the two-way table of `sections` (rows) by `period` (columns) with its margins. Then read from it `essays_late`, the number of essays published in 2019–2025, and `late_total`, the number of articles of any section published in 2019–2025.

In [ ]:
table = ...
essays_late = ...
late_total = ...

check.ex4(table, essays_late, late_total)

<details>
<summary>💡 Hint</summary>

Copy the `pd.crosstab` call above. A cell is read with `.loc[row, column]`, as in `table.loc["Essays", "2019-2025"]`; the totals are in the row called `"All"`.

</details>

<details>
<summary>✅ Solution</summary>

```python
table = pd.crosstab(sections, period, margins=True)
essays_late = table.loc["Essays", "2019-2025"]
late_total = table.loc["All", "2019-2025"]

check.ex4(table, essays_late, late_total)
```

</details>

## 6 · Row and column percentages

A two-way table can be turned into percentages in two ways, and they answer different questions.

- **Row percentages** divide each cell by its row total: $n_{ij} / n_{i\cdot}$. They compare the rows. *Among the articles not in Italian, what share are essays?* $27 / 31 = 87\%$; among the Italian ones, $142 / 290 = 49\%$.
- **Column percentages** divide each cell by its column total: $n_{ij} / n_{\cdot j}$. They compare the columns. *Among the essays, what share are not in Italian?* $27 / 169 = 16\%$; among the other sections, $4 / 152 = 3\%$.

Both start from the same cell, the 27 essays not in Italian, but divide it by different totals. "87% of the articles in other languages are essays" is true. "87% of the essays are in other languages" is false: only 16% are. Mixing up the two directions is one of the commonest errors in reading statistics. Part 2 meets it again as the difference between two conditional probabilities, and lesson S2.4 shows how one can be turned into the other.

The two readings, as bars that each add up to 100%:

In [ ]:
lang = pd.Series(np.where(languages == "Italian", "Italian", "other language"),
                 index=languages.index, name="language")
genre = pd.Series(np.where(sections == "Essays", "essay", "other section"),
                  index=sections.index, name="section")
fig, axes = plt.subplots(1, 2, figsize=(9, 3.6))
pd.crosstab(lang, genre, normalize="index").plot.barh(
    stacked=True, ax=axes[0], color=[plots.BLUE, plots.MUTED])
axes[0].set(title="row percentages: within each language", xlabel="share")
pd.crosstab(genre, lang, normalize="index").plot.barh(
    stacked=True, ax=axes[1], color=[plots.BLUE, plots.MUTED])
axes[1].set(title="column percentages: within each section", xlabel="share")
for ax in axes:                                  # legends below the bars
    ax.legend(ncol=2, loc="upper center", bbox_to_anchor=(0.5, -0.25),
              frameon=False)
fig.tight_layout()
plt.show()

The right panel shows the column percentages of the table as rows of bars: for each kind of section, the share of each language. **In code**, `pd.crosstab(..., normalize="index")` gives row shares, and `normalize="columns"` gives column shares.

### ✏️ Exercise 5 · Which way round?

Using `lang` and `genre`, compute `essays_among_other`, the share of essays among the articles not in Italian; `other_among_essays`, the share of articles not in Italian among the essays; and `essays_among_italian`, the share of essays among the Italian articles. Give shares between 0 and 1.

In [ ]:
shares_by_row = pd.crosstab(lang, genre, normalize="index")
shares_by_col = pd.crosstab(lang, genre, normalize="columns")
essays_among_other = ...
other_among_essays = ...
essays_among_italian = ...

check.ex5(essays_among_other, other_among_essays, essays_among_italian)

<details>
<summary>💡 Hint</summary>

Print the two tables first. In `shares_by_row` each row adds up to 1, so it answers "among the articles in this language". In `shares_by_col` each column adds up to 1, so it answers "among the articles in this section". Read the cells with `.loc[row, column]`, as in `shares_by_row.loc["other language", "essay"]`.

</details>

<details>
<summary>✅ Solution</summary>

```python
shares_by_row = pd.crosstab(lang, genre, normalize="index")
shares_by_col = pd.crosstab(lang, genre, normalize="columns")
essays_among_other = shares_by_row.loc["other language", "essay"]
other_among_essays = shares_by_col.loc["other language", "essay"]
essays_among_italian = shares_by_row.loc["Italian", "essay"]

check.ex5(essays_among_other, other_among_essays, essays_among_italian)
```

</details>

## 7 · Counting references by type

The syllabus asks for references by type, but the references table has no type column: each reference is a line of text. To count types, we must first decide what a type is and how to recognise it, which is the operationalisation of S1.1. Italian citation style helps: journal titles are written between guillemets, «…», and parts of a larger volume are introduced by "in". Four rules, applied in order:

In [ ]:
def ref_type(text):
    if "«" in text:                          # a title in guillemets: a periodical
        return "periodical"
    if ", in " in text:                      # part of a larger volume
        return "chapter"
    if "http" in text or "www." in text:     # a web address
        return "web"
    return "book or other"

types = bib.references.set_index("reference")["content"].map(ref_type)
types = types.rename("type")
types.value_counts().to_frame("references")

The `if` lines check the rules in order, and `return` gives the answer as soon as one rule matches. `.map(ref_type)` applies the function to every reference. Before trusting the counts, read a few references of each type:

In [ ]:
import textwrap                              # to wrap long lines of text

for kind in ["chapter", "web"]:
    print(f"— {kind}:")
    for text in bib.references["content"][types.to_numpy() == kind].sample(
            3, random_state=2):
        print(textwrap.fill(text[:230], 88, initial_indent="  • ",
                            subsequent_indent="    "))

Most are right, but not all. Some references counted as web are printed books and inventories whose reference simply adds a link to a digitised copy, such as an inventory printed in 1988: the rule counts where the reference points, not what the work is. A Latin title containing ", in" would be counted as a chapter. These counts are only as valid as the rules, and the rules are rough. A study that needed reliable types would check a random sample by hand and report how often the rules were wrong.

**Types over time.** With the year of the citing article (S0.2's citation links), a two-way table compares the types cited in 2016–2020 and in 2021–2025. Column percentages compare the two periods, which have different numbers of citations:

In [ ]:
links = bibliothecae.citation_links(bib)          # one row per citation
links["type"] = links["reference"].map(types)
links["period"] = np.where(links["citing_year"] <= 2020, "2016-2020", "2021-2025")
pd.crosstab(links["type"], links["period"], margins=True)

### ✏️ Exercise 6 · More chapters?

Compute `share_web`, the share of all the journal's references (in `types`) that the rules call web. Then, from the citations in `links`, compute `pct_chapter_early` and `pct_chapter_late`, the percentage of chapters among the citations of 2016–2020 and of 2021–2025, and `change`, the difference between the two in percentage points (S0.1), later minus earlier.

In [ ]:
by_period = pd.crosstab(links["type"], links["period"], normalize="columns") * 100
share_web = ...
pct_chapter_early = ...
pct_chapter_late = ...
change = ...

check.ex6(share_web, pct_chapter_early, pct_chapter_late, change)

<details>
<summary>💡 Hint</summary>

`types.value_counts(normalize=True)` gives the shares of the types, and `["web"]` picks one. `by_period` already holds column percentages: each column adds up to 100. Read a cell with `.loc["chapter", "2016-2020"]`, and subtract the two percentages for the change in points.

</details>

<details>
<summary>✅ Solution</summary>

```python
by_period = pd.crosstab(links["type"], links["period"], normalize="columns") * 100
share_web = types.value_counts(normalize=True)["web"]
pct_chapter_early = by_period.loc["chapter", "2016-2020"]
pct_chapter_late = by_period.loc["chapter", "2021-2025"]
change = pct_chapter_late - pct_chapter_early

check.ex6(share_web, pct_chapter_early, pct_chapter_late, change)
```

Chapters rose by about 3 points, periodicals fell by about 3, and web references stayed at about 4% in both periods. Whether a 3-point change in counts this size is more than chance is a question for Part 4.

</details>

> **🔬 Metascience.** Counting what scholars cite is one of the oldest uses of bibliometrics in library science, because it tells libraries what to collect. Comparing eight humanities fields, Jennifer Knievel and Charlene Kellsey found that "in most humanities disciplines, monographs remain the dominant format of cited sources" ([Knievel & Kellsey, 2005](https://doi.org/10.1086/431331)). By the rough rules above, *Bibliothecae\.it* fits the pattern: references that are not periodicals (books, chapters and other documents) make up nearly three quarters of the total, and periodicals about a quarter. Such comparisons are only as good as the classification behind them, and studies classify in different ways, so the method has to be read before the numbers.

> **🤖 Why this matters for AI.** Frequency tables are where language models begin: the word chances of S0.2 were relative frequencies, $f_k = n_k / n$. A classifier that sorts references into types gives a score to each type, and its answer is the argmax of the scores. Its mistakes are then counted in a two-way table called a **confusion matrix**, with the true types as rows and the predicted types as columns. The two directions of this lesson become two standard measures. **Recall** is a row percentage: among the references that really are web, what share did the classifier call web? **Precision** is a column percentage: among the references it called web, what share really are web? A classifier can be strong on one and weak on the other, exactly as 87% and 16% above came from one cell. Lessons S2.4 and S3.7, and module M2, use both.

## 8 · Wrap-up

### ✍️ Explain it

A colleague reads the two-way table of section 5 and writes: "Since 87% of the journal's articles in other languages are essays, the journal's essays are mostly in other languages." In two or three sentences, explain the mistake, and say which number answers the colleague's question.

Write your answer in the next cell.

*Your answer:*

<details>
<summary>A model answer</summary>

The 87% is a row percentage: it divides the 27 essays not in Italian by the 31 articles not in Italian, so it describes the articles in other languages. The question is about the essays, so it needs the column percentage, which divides the same 27 by the 169 essays: only 16% of the essays are in other languages, and 84% are in Italian. The same cell gives two different shares depending on which total it is divided by.

</details>

### The words and symbols of this lesson

| Term or symbol | Read it as | Means |
| --- | --- | --- |
| $n_k$, $f_k = n_k / n$ | "n k", "f k" | the count and the share of category k |
| $F_k = \sum_{j \le k} f_j$ | "capital F k" | the cumulative share up to category k |
| $\arg\max_k n_k$ | "the argmax over k of n k" | the category with the largest count: the mode |
| $n_{ij}$ | "n i j" | the count in row i and column j of a two-way table |
| $n_{i\cdot}$, $n_{\cdot j}$ | "n i dot", "n dot j" | a row total, a column total: the margins |
| $n_{ij} / n_{i\cdot}$ | | a row percentage: compares the rows |
| $n_{ij} / n_{\cdot j}$ | | a column percentage: compares the columns |

### Check your progress

In [ ]:
check.summary();

### Done when

- [ ] Exercises 1 to 6 show ✅ in `check.summary()`.
- [ ] You can build a frequency table with counts, shares and cumulative shares for any column.
- [ ] You can say why bars must start at zero.
- [ ] Given a two-way table and a question, you can say whether the question needs row or column percentages.

### Further reading

- Cleveland, W. S., & McGill, R. (1984). Graphical perception: Theory, experimentation, and application to the development of graphical methods. *Journal of the American Statistical Association*, 79(387), 531–554. <https://www.jstor.org/stable/2288400>
- Knievel, J. E., & Kellsey, C. (2005). Citation analysis for collection development: A comparative study of eight humanities fields. *The Library Quarterly*, 75(2), 142–168. <https://doi.org/10.1086/431331>

**Next:** S1.3 · Distributions and their shape.